<a href="https://colab.research.google.com/github/guitorte/audio/blob/main/music-to-midi/notebooks/Assemble_MIDI_to_REMI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧩 Montar MIDI + texto REMI (MIDI-TEXT v2)

Junta os MIDIs de cada parte de uma música num arquivo só e escreve a música como **texto MIDI-TEXT v2 (REMI-style)**, o mesmo formato do notebook *Song → Stems → MIDI v2*:

```
KEY: E major
TEMPO_BPM: 86.13
CHORDS: 3.11:Em | 5.87:Em | 8.66:E | …
TRACK_BEGIN
NAME: vocals
NOTE	3	9	12.981	0.049	G2	43	64      ← compasso, 16ª, início (s), duração (s), nota, MIDI, velocity
```

**O que entra** (uma pasta ou um `.zip` no Drive, ou enviado na hora):
- as partes: os `.mid` do bake-off (`per_stem/`, nomes como `bs_roformer_sw__basic_pitch__bass.mid`) e/ou arquivos com nomes simples (`bass.mid`, `musica_bateria.mid`…);
- a voz: um arquivo terminado em `_vocals.mid` (como o do Vocal Lab) tem prioridade sobre as vozes do bake-off.

Também aceita **um MIDI já montado** (o `.mid` de uma música que você já juntou, ou um da sua DAW): aí ele só gera o texto, sem mexer no MIDI.

**O que sai**, em `<pasta de músicas>/midi-montado/<música>/`:

| Arquivo | O que é |
|---|---|
| `<música>.mid` | todas as partes, com a voz em destaque (lead quadrado) e volume/pan por faixa |
| `<música>_instrumental.mid` | o mesmo, sem a voz |
| `<música>.remi.txt` | o texto MIDI-TEXT v2, com as velocities originais das transcrições |
| `<música>_instrumental.remi.txt` | o texto sem a voz |

O compasso, o tempo, a tonalidade e os acordes vêm do **áudio da música** quando você indica o arquivo (como no notebook v2). Sem o áudio, vêm de um render do próprio MIDI. Tonalidade e acordes são estimativas nos dois casos.

In [ ]:
#@title 1 · Instalar (≈ 30 s)
# FluidSynth + soundfont General MIDI: para analisar o MIDI sem o áudio e para a prévia.
!apt-get -qq install -y fluidsynth fluid-soundfont-gm > /dev/null
!pip -q install pretty_midi mido
print("✓ instalado")

In [ ]:
#@title 2 · Conectar o Drive e listar o que há na pasta
DRIVE_FOLDER = "áudio"  #@param {type:"string"}

import os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

REPO, BRANCH = "/content/audio", "main"
if not os.path.isdir(REPO):
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH,
                       "https://github.com/guitorte/audio.git", REPO]).returncode:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/guitorte/audio.git", REPO], check=True)
sys.path.insert(0, f"{REPO}/music-to-midi")
from modules import remi as R
U = R.U

FOLDER = U.find_folder(DRIVE_FOLDER)
SOURCES = R.list_sources(FOLDER)
AUDIOS = U.list_audio(FOLDER)
print(f"📁 {FOLDER}\n\nMIDIs (pasta, .zip ou .mid):")
for i, p in enumerate(SOURCES, 1):
    print(f"{i:3d}. {p.relative_to(FOLDER)}{'/' if p.is_dir() else ''}")
print("\nÁudios:")
for i, p in enumerate(AUDIOS, 1):
    print(f"{i:3d}. {p.name}")

In [ ]:
#@title 3 · Montar e gerar o texto
SOURCE = ""  #@param {type:"string"}
#@markdown Número da lista de MIDIs acima, parte do nome, ou um caminho. **Vazio = enviar agora** (um `.zip` ou vários `.mid`).
AUDIO = ""  #@param {type:"string"}
#@markdown Opcional: número ou nome do áudio da música (lista acima). Com ele, compasso, tempo, tonalidade e acordes vêm da música.
SONG_NAME = ""  #@param {type:"string"}
#@markdown Vazio = o nome do arquivo de voz sem `_vocals` (ou o nome da pasta/zip).
DRUMS = "adt_str"  #@param ["adt_str", "adtof"]
#@markdown Qual bateria usar quando houver as duas.
VELOCITY_BOOST = 1.35  #@param {type:"slider", min:1.0, max:1.6, step:0.05}
#@markdown Deixa o `.mid` mais alto (1.0 = velocities originais). O texto sempre guarda as originais.
OVERWRITE = False  #@param {type:"boolean"}

import shutil
WORK = Path("/content/remi_work")
if SOURCE.strip():
    src = R.resolve(SOURCE, FOLDER, SOURCES)
else:
    from google.colab import files
    up = files.upload()
    shutil.rmtree(WORK / "upload", ignore_errors=True)
    (WORK / "upload").mkdir(parents=True)
    for name, data in up.items():
        (WORK / "upload" / name).write_bytes(data)
    zips = [p for p in (WORK / "upload").iterdir() if p.suffix.lower() == ".zip"]
    src = zips[0] if len(up) == 1 and zips else WORK / "upload"
audio = R.resolve(AUDIO, FOLDER, AUDIOS) if AUDIO.strip() else None

OUT = R.process(src, FOLDER / "midi-montado", SONG_NAME.strip() or None, audio,
                DRUMS, VELOCITY_BOOST, OVERWRITE, WORK)
print()
for k, p in OUT.items():
    print(f"  {p.relative_to(FOLDER)}")

from IPython.display import Audio, display
txt = OUT["remi"].read_text(encoding="utf-8").splitlines()
print("\n" + "\n".join(l if len(l) < 160 else l[:157] + "…" for l in txt[2:16]))
if "mid" in OUT:
    import pretty_midi
    wav = R.render(pretty_midi.PrettyMIDI(str(OUT["mid"])), WORK / "preview.wav")
    display(Audio(str(wav)))

## Observações

* **Quais arquivos ele escolhe:** uma voz `_vocals.mid` ganha das vozes do bake-off (e, entre essas, o GAME ganha do Basic Pitch). Stems do BS-RoFormer SW ganham dos do htdemucs_6s. A bateria segue o `DRUMS`. As faixas do MuScriptor e os arquivos "A_atual…", "C_RoFormer_SW…" (as montagens do próprio bake-off) ficam de fora. A célula 3 lista o que escolheu e o que deixou de lado.
* **O tempo do `.mid`** é o do arquivo de voz (o Vocal Lab grava o tempo detectado). Se nenhum arquivo tiver um, usa o tempo detectado no áudio ou no render.
* **O texto volta ao MIDI sem perda**: as notas, velocities e tempos (ao milissegundo) são conferidos depois de escrever. Compasso e posição (`bar`, `pos16`) servem para ler e editar; `start_s` e `midi` são os que valem.
* **Arquivos que já existem** na pasta de saída são mantidos, a menos que `OVERWRITE` esteja marcado.
* Depois de um `git push` com correções, rode *Runtime ▸ Disconnect and delete runtime* antes de usar de novo, senão o Colab continua com o código antigo em `/content/audio`.